In [15]:
import numpy as np
import pickle 
import pandas as pd
import heapq

In [14]:
game = np.arange(1,9)
game = np.append(game,0)
final_state = game.reshape((3,3))
np.random.shuffle(game)
game = game.reshape((3,3))
print(game)

[[3 2 4]
 [6 1 8]
 [7 0 5]]


In [16]:
# we first need to make a node class to amke sure we confer to the 
# rules of the game 


class Node:
    def __init__(self,board,level,fval):
        self.board = board
        self.level = level
        self.fval= fval 
        
    

In [20]:
import numpy as np
import random

def initialize_board():
    board = np.zeros((8, 8), dtype=int)
    for r in range(3):
        for c in range(8):
            if (r + c) % 2 == 1: board[r, c] = -1
    for r in range(5, 8):
        for c in range(8):
            if (r + c) % 2 == 1: board[r, c] = 1
    return board

def calculate_reward(old_board, new_board, player, game_over, winner):
    if game_over:
        return 100 if winner == player else -100
    old_enemy = np.sum(np.sign(old_board) == -player)
    new_enemy = np.sum(np.sign(new_board) == -player)
    if new_enemy < old_enemy:
        return 10  # reward
    return -1      # penalty


class SimpleCheckersAgent:
    def __init__(self):
        # [peices, kings, cneter?, bias]
        self.weights = np.array([1.0, 2.0, 0.5, 1.0])
        self.alpha, self.gamma, self.epsilon = 0.05, 0.95, 0.3

    def get_features(self, board, player):
        my_pieces = np.sum(board == player)
        my_kings = np.sum(board == player * 2)
        center_control = np.sum(board[2:6, 2:6] == player)
        return np.array([float(my_pieces), float(my_kings), float(center_control), 1.0])

    def get_q_value(self, features):
        return np.dot(self.weights, features)

    def select_action(self, board, legal_moves, player):
        if not legal_moves: return None
        if random.random() < self.epsilon:
            return random.choice(legal_moves)
        qs = [self.get_q_value(self.get_features(self.simulate_move(board, m), player)) for m in legal_moves]
        return legal_moves[np.argmax(qs)]

    def simulate_move(self, board, move):
        temp_board = board.copy()
        (r1, c1), (r2, c2) = move
        temp_board[r2, c2] = temp_board[r1, c1]
        temp_board[r1, c1] = 0
        if abs(r2 - r1) == 2: # capture
            temp_board[(r1 + r2) // 2, (c1 + c2) // 2] = 0
        if r2 == 0 and temp_board[r2, c2] == 1: # kinging
            temp_board[r2, c2] = 2
        return temp_board

    def update_weights(self, s_feats, reward, s_prime_feats, done):
        current_q = self.get_q_value(s_feats)
        target = reward if done else reward + self.gamma * self.get_q_value(s_prime_feats)
        self.weights += self.alpha * (target - current_q) * s_feats


agent = SimpleCheckersAgent()
num_episodes = 2000

for ep in range(num_episodes):
    board = initialize_board()
    done = False
    while not done:
        # player 1 turn
        moves = get_legal_moves(board, 1) # defined in previous step
        if not moves: break
        
        s_feats = agent.get_features(board, 1)
        action = agent.select_action(board, moves, 1)
        next_board = agent.simulate_move(board, action)
        
        reward = calculate_reward(board, next_board, 1, False, None)
        
        # opponent turn (random)
        opp_moves = get_legal_moves(next_board, -1)
        if not opp_moves:
            done = True #  win
            reward = 100
            
        else:
            opp_action = random.choice(opp_moves)
            next_board = agent.simulate_move(next_board, opp_action) # Basic reuse of logic
        
        s_prime_feats = agent.get_features(next_board, 1)
        agent.update_weights(s_feats, reward, s_prime_feats, done)
        board = next_board


print(f"trained weights {agent.weights}")
print(f"Interpretation: Piece Val={agent.weights[0]:.2f}, King Val={agent.weights[1]:.2f}, Center={agent.weights[2]:.2f}")
print("-" * 30)

trained weights [ 6.68687562 15.57102205 -0.23094641 77.16304821]
Interpretation: Piece Val=6.69, King Val=15.57, Center=-0.23
------------------------------
